# PROJETO 1 ANADI - E-REDES

In [ ]:
# TODOS OS IMPORTS SÃO FEITOS AQUI PARA MANTER SIMPLICIDADE
# BIBLIOTECAS DISPONÍVEIS:
#   - matplotlib
#   - pandas
#   - numpy
#   - scipy
#   - statsmodels

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

ip_data = pd.read_excel("./data/IP_data.xlsx")
ptd_data = pd.read_excel("./data/PTD_data.xlsx")

### 1- Processamento da Iluminação Pública (IP_Data)

In [ ]:
def is_inefficient(type: str) -> int:
    """
    Uma lâmpada é considerada ineficente se for de sódio
    ou de mercúrio 
    """
    return int(type in ["Sódio", "Mercúrio"]);

# Variavel `Is_Ineficiente`, indica e eficiencia de uma lampada
ip_data["Is_Ineficiente"] = ip_data["Tipo de Lâmpada"].apply(is_inefficient);

In [ ]:
def power(total_power: float) -> float:
    """
    Calcula a potência em kW
    Para tal recebe a potência total instalada (em W) e divide a por 1000
    P = (pW / 1000) / 1000
    """
    return total_power / 10000;

ip_data["Pot_Kw"] = ip_data["Potência Instalada Total (W)"].apply(power);

In [ ]:
# Soma da Potência kW de todas as linhas de IP do concelho
p_ip_total = ip_data.groupby("CodDistritoConcelho")["Pot_Kw"].sum();

# Soma da Potência kW das linhas onde Is_Ineficiente == 1
p_ip_inef = (
    ip_data[ip_data["Is_Ineficiente"] == 1]
    .groupby("CodDistritoConcelho")["Pot_Kw"]
    .sum()
);

result = pd.DataFrame({
    "P_IP_Total": p_ip_total,
    "P_IP_Inef": p_ip_inef
}).fillna(0);

print(result)

### 2- Processamento dos Postos de Transformação (PTD data)

In [ ]:
def parse_usage(usage: str) -> float:
    """
    Converter o nível de utilização num número decimal
    Exemplo: '60%-79%' -> 0.79
    Pelo que consigo ver o formato pode tomar os seguintes formatos:
    - 'X%-Y%'
    - +100%
    - N/D
    Não temos instruções de o que fazer por isso vou assumir que N/D -> NaN
    """
    if usage == "+100%":
        return 1;

    if '-' in usage:
        last_number: str = usage.split("-")[1].removesuffix("%");
        return int(last_number) / 100;

    return np.nan;

ptd_data["Nível de Utilização [%]"] = ptd_data["Nível de Utilização [%]"].apply(parse_usage);

In [ ]:
# Cap_PTD: Soma da potência instalada [kVA] de todos os PTDs do concelho
# Util_Media: Média do nível de utilização dos PTDs do concelho
# N_PTDs: Número de PTDs nesse concelho
ptd_stats = ptd_data.groupby("CodDistritoConcelho").agg(
    Cap_PTD=("Potência instalada [kVA]", "sum"),
    Util_Media=("Nível de Utilização [%]", "mean"),
    N_PTDs=("CodDistritoConcelho", "count")
);

print(ptd_stats)

### 3- Variáveis do Novo Dataset

Com os dados calculados anteriormente, vamos definir um novo dataset com:
- Ganho LED (∆PLED ): Representa a potência a ser libertada pela substituição de lâmpadas ineficientes por tecnologia LED.
- Folga Rede (PFolga): Estima a capacidade disponível na rede, descontando a utilização atual e aplicando uma margem de segurança de 92%.
- Carga VE (PVE ): Projeta o aumento de carga necessário para instalar carregadores de 22 kW em todos os PTDs do concelho.
- Saldo Final de Viabilidade (D): O indicador que subtrai a nova carga (PVE) à capacidade total disponível (PFolga + ∆PLED ), determinando se o projeto é viável.
- Rate Ineficiencia: O rácio que mede o peso da tecnologia obsoleta face ao total da iluminação do concelho.

In [ ]:
# Create an empty DataFrame using the municipalities from p_ip_inef as rows
resulting_data: pd.DataFrame = pd.DataFrame(index=p_ip_inef.index)

In [ ]:
LED_SAVINGS_FACTOR = 0.65         # 65% reduction when replacing inefficient lamps
GRID_MARGIN = 0.92                # 92% of transformer capacity is considered usable
EV_CHARGER_POWER = 22             # kW per EV charger
EV_SIMULT_FACTOR = 0.6            # simultaneity factor for EV chargers

resulting_data["Ganho LED"] = p_ip_inef * LED_SAVINGS_FACTOR
resulting_data["Folga Rede"] = (ptd_stats["Cap_PTD"] * GRID_MARGIN) * (1 - ptd_stats["Util_Media"])
resulting_data["Carga VE"] = ptd_stats["N_PTDs"] * EV_CHARGER_POWER * EV_SIMULT_FACTOR
resulting_data["Saldo Final de Viabilidade"] = (
    resulting_data["Folga Rede"] + resulting_data["Ganho LED"] - resulting_data["Carga VE"]
)
resulting_data["Rate Ineficiencia"] = p_ip_inef / p_ip_total

print(resulting_data)

### 4.3 - Análise e Exploração de Dados

1- Construa uma representação gráfica que permita visualizar o mix tecnológico (LED vs. Convencional). Identifique se a maioria da potência ineficiente se concentra num grupo restrito de municípios.

In [ ]:
fig, ax = plt.subplots(3, 1)

inefficient = np.array(ip_data[ip_data["Is_Ineficiente"] == 1].groupby("CodDistrito")["Luminárias"].sum())
efficient = np.array(ip_data[ip_data["Is_Ineficiente"] == 0].groupby("CodDistrito")["Luminárias"].sum())

# I can't stress enough how impossible it is to turn the below array of "stringarrays" into a normal array of strings.
#labels = np.array(ip_data[ip_data["Is_Ineficiente"] == 1].groupby("CodDistrito")["Distrito"].unique())
# So we're doing this instead.
labels = ["Aveiro", "Beja", "Braga", "Bragança", "Castelo Branco", "Coimbra", "Évora", "Faro", "Guarda", "Leiria", "Lisboa", "Portalegre", "Porto", "Santarém", "Setúbal", "Viana do Castelo", "Vila Real", "Viseu"]
joined = np.concatenate((inefficient, efficient))

tab20 = plt.color_sequences["tab20"]
ax[0].pie([np.sum(inefficient), np.sum(efficient)], labels=["Ineficiente", "Eficiente"], radius=1.3)
ax[0].set_title("Mix tecnológico", fontsize=10)
ax[1].pie(inefficient, colors=tab20, radius=1.3)
ax[1].set_title("IP Ineficiente por Distrito", fontsize=10)
wedges, texts = ax[2].pie(efficient, colors=tab20, radius=1.3)
ax[2].set_title("IP Eficiente por Distrito", fontsize=10)

ax[0].legend(wedges, labels, title="Distritos", bbox_to_anchor=(1.7, 0, 0, 1.4))

plt.show()

Os dados grafados acima demonstram muito claramente que a potência usada em iluminação ineficiente não é concentrada desproporcionalmente em um grupo particular de municípios: É distribuída, de um modo muito semelhante á distribuição da potência usada em iluminação eficiente, ao longo do país.

2- Construa boxplots para comparar as distribuições dos níveis de utilização dos PTDs entre os distritos de Lisboa, Porto, Aveiro e Setúbal. Identifique o distrito com maior variabilidade de carga.


In [ ]:
# Limpar os NaNs primeiro para não dar erro no matplotlib
ptd_clean = ptd_data.dropna(subset=["Nível de Utilização [%]"])

# Extrair os arrays por distrito
lisboa_data = ptd_clean[(ptd_clean["CodDistritoConcelho"] >= 1100) & (ptd_clean["CodDistritoConcelho"] < 1200)]["Nível de Utilização [%]"].to_numpy()
porto_data = ptd_clean[(ptd_clean["CodDistritoConcelho"] >= 1300) & (ptd_clean["CodDistritoConcelho"] < 1400)]["Nível de Utilização [%]"].to_numpy()
aveiro_data = ptd_clean[(ptd_clean["CodDistritoConcelho"] >= 100) & (ptd_clean["CodDistritoConcelho"] < 200)]["Nível de Utilização [%]"].to_numpy()
setubal_data = ptd_clean[(ptd_clean["CodDistritoConcelho"] >= 1500) & (ptd_clean["CodDistritoConcelho"] < 1600)]["Nível de Utilização [%]"].to_numpy()

# Agrupar os dados e criar os labels
dados_grafico = [aveiro_data, lisboa_data, porto_data, setubal_data]
labels = ['Aveiro', 'Lisboa', 'Porto', 'Setúbal']

# Criar o Boxplot (Apenas UMA figura)
plt.figure(figsize=(10, 6))

plt.boxplot(dados_grafico, patch_artist=True, 
            boxprops=dict(facecolor="lightblue"),
            flierprops=dict(marker='o', markerfacecolor='red', markersize=5, alpha=0.5))

plt.title('Distribuição dos Níveis de Utilização dos PTDs por Distrito')
plt.ylabel('Nível de Utilização (Escala Decimal)')
plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.show()

# Identificar o distrito com MAIOR variabilidade de carga (Desvio Padrão)
variabilidade = {
    'Aveiro': np.std(aveiro_data, ddof=1),
    'Lisboa': np.std(lisboa_data, ddof=1),
    'Porto': np.std(porto_data, ddof=1),
    'Setúbal': np.std(setubal_data, ddof=1)
}

# Encontrar distrito com o maior valor no dicionário
distrito_max_var = max(variabilidade, key=variabilidade.get)

print("--- Variabilidade de Carga (Desvio Padrão) ---")
for dist, var in variabilidade.items():
    print(f"{dist}: {var:.4f}")

print(f"\nConclusão: O distrito com maior variabilidade de carga é {distrito_max_var.upper()} com um desvio padrão de {variabilidade[distrito_max_var]:.4f}.")

TypeError: boxplot() got an unexpected keyword argument 'tick_labels'

<Figure size 1000x600 with 0 Axes>

3- Quantifique a prevalência de valores omissos ou indeterminados (N/D, <20). Utilize uma representação gráfica para identificar a presença de outliers nos níveis de ocupação da rede.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Conta os omissos (N/D) que ja foram convertidos para NaN
omissos_nd = ptd_data['Nível de Utilização [%]'].isna().sum() 

# Conta os que indeterminados, ou seja, são < 0.20
menorque_20 = ptd_data[ptd_data['Nível de Utilização [%]'] < 0.20].shape[0]

# Calcular a Prevalência
total_registos = len(ptd_data)
total_anomalos = omissos_nd + menorque_20
prevalencia = (total_anomalos / total_registos) * 100

print(f"--- PREVALÊNCIA DE VALORES OMISSOS/INDETERMINADOS ---")
print(f"Total de PTDs: {total_registos}")
print(f"Valores Omissos (N/D): {omissos_nd}")
print(f"Valores Indeterminados (<20): {menorque_20}")
print(f"Prevalência: {prevalencia:.2f}% da rede ({total_anomalos} postos)\n")


# 4. Boxplot para identificar Outliers
plt.figure(figsize=(10, 4))

# Criamos o boxplot global
ptd_data.boxplot(column='Nível de Utilização [%]', vert=False, grid=False,
                 patch_artist=True, boxprops=dict(facecolor="lightblue"),
                 flierprops=dict(marker='o', markerfacecolor='red', markersize=6, alpha=0.6))

plt.title('Identificação de Outliers nos Níveis de Ocupação da Rede')
plt.xlabel('Nível de Utilização (escala decimal)')
plt.yticks([1], ['Todos os PTDs'])

plt.show()

4- Construa uma tabela com a média, quartis, desvio padrão, assimetria e curtose do nível de utilização médio para os concelhos de Coimbra, Évora, Braga e Faro (apresente com 4 casas decimais).

In [ ]:
# 4. Função para calcular todas as estatísticas pedidas (Corrigido)
tabela_estatisticas = df_concelhos.groupby('Concelho_Upper')['Nível de Utilização [%]'].agg(
    Média='mean',
    Q1=lambda x: x.quantile(0.25),
    Mediana='median',
    Q3=lambda x: x.quantile(0.75),
    Desvio_Padrão='std',
    Assimetria='skew',
    Curtose=lambda x: x.kurt()
)

# 5. Formatar os resultados para 4 casas decimais, como pede o enunciado
tabela_estatisticas = tabela_estatisticas.round(4)

print("--- Estatísticas Descritivas do Nível de Utilização dos PTDs ---")
display(tabela_estatisticas)

### 4.4 - Inferência Estatística

1- Considere a base de dados consolidada e selecione aleatoriamente uma amostra de 50 concelhos. Use esta amostra para testar se o nível médio de ocupação da rede é inferior a um patamar de referência (ex: 60\%), verificando previamente a normalidade dos dados.

**Nota:** Vamos definir cada teste como a sua própria função com os seus argumentos de testes para podermos facilmente repetir os testes e validar padrões

In [ ]:
def test_normalidade(dados: pd.Series, nome: str = "", alpha: float = 0.05, verbose: bool = True) -> bool:
    """
    Verifica a normalidade de uma amostra visualmente e com o teste de Shapiro-Wilk.
    Retorna True se os dados seguem distribuição normal, False caso contrário.
    """
    if verbose:
        # Verificar a normalidade de maneira visual (https://thalesferraz.medium.com/testes-de-normalidade-usando-python-1abddbc9311f)
        fig, ax = plt.subplots(figsize=(6, 4));
        dados.plot(kind='hist', density=True, ax=ax);
        dados.plot(kind='kde', ax=ax);
        ax.set_title(f"Distribuição - {nome}");
        ax.set_xlabel("Valor");
        plt.show();

    shapiro_result = stats.shapiro(dados);
    stat = shapiro_result.statistic;
    p_val = shapiro_result.pvalue;

    if verbose:
        print(f"Teste de Shapiro-Wilk: {nome}");
        print(f"Estatística: {shapiro_result.statistic}, p-value: {p_val}");
        if p_val > alpha:
            print("Não se rejeita H0: dados seguem distribuição normal\n");
        else:
            print("Rejeita-se H0: dados não seguem distribuição normal\n");

    return p_val > alpha;

In [ ]:
def teste_ocupação_media_menor_que(p: float = 0.60, s: int = 32, alpha: float = 0.05, verbose: bool = True) -> bool:
    """
    Testa se o nível médio de ocupação da rede é inferior a um patamar de referência.
    Usa uma amostra aleatória de 50 concelhos do dataset ptd_stats.
    Retorna True se não se rejeita H0 (ocupação >= p), False caso contrário.
    """
    amostra_concelhos = ptd_stats.sample(n=50, random_state=s);
    dados_ocupacao = amostra_concelhos['Util_Media'];

    # Verificar normalidade dos dados antes de aplicar o teste t
    norm = test_normalidade(dados_ocupacao, verbose=verbose, alpha=alpha);
    if not norm:
        raise ValueError("Os dados deixaram de ser normais!");

    # H0: media >= p (A ocupação não é inferior ao patamar)
    # H1: media < p  (A ocupação é significativamente inferior ao patamar)
    statistic, pvalue = stats.ttest_1samp(dados_ocupacao, p, alternative='less');

    if verbose:
        print(f"\nResultado do Teste:");
        print(f"Média da Amostra: {dados_ocupacao.mean()}");
        print(f"Estatística t: {statistic}");
        print(f"p-value: {pvalue}");
        if pvalue < alpha:
            print("Rejeita-se H0: ocupação média é inferior a 60%");
        else:
            print("Não se rejeita H0: não há evidência de que ocupação < 60%");

    return pvalue >= alpha;

teste_ocupação_media_menor_que();

2- Selecione aleatoriamente duas amostras de 30 registos: uma de concelhos "Modernizados" (rácio de LED acima da mediana) e outra de concelhos "Ineficientes". Use estas amostras para testar se o estado médio de ocupação da rede difere significativamente entre os dois grupos.

In [ ]:
def teste_ocupação_media_ef_vs_nef(s: int = 32, alpha: float = 0.05, verbose: bool = True) -> bool:
    """
    Testa se o nível médio de ocupação da rede difere significativamente entre
    concelhos Modernizados (rácio LED acima da mediana) e Ineficientes.
    Usa amostras aleatórias de 30 concelhos de cada grupo.
    Retorna True se não se rejeita H0 (não há diferença), False caso contrário.
    """
    # Juntar Util_Media ao resulting_data para ter tudo num só DataFrame
    combined = resulting_data.join(ptd_stats['Util_Media']);

    # Rate_Ineficiencia abaixo da mediana (menos ineficientes = mais LED)
    mediana_inef = combined['Rate Ineficiencia'].median();
    modernizados = combined[combined['Rate Ineficiencia'] <= mediana_inef];
    ineficientes = combined[combined['Rate Ineficiencia'] > mediana_inef];

    # Amostras de 30 de cada grupo
    amostra_mod  = modernizados.sample(n=30, random_state=s)['Util_Media'];
    amostra_inef = ineficientes.sample(n=30, random_state=s)['Util_Media'];

    # Verificar normalidade de ambos os grupos antes de escolher o teste
    normal_mod  = test_normalidade(amostra_mod,  nome="Modernizados",  alpha=alpha, verbose=verbose);
    normal_inef = test_normalidade(amostra_inef, nome="Ineficientes",  alpha=alpha, verbose=verbose);

    if not normal_mod or not normal_inef:
        raise ValueError("Os dados deixaram de ser normais!");

    # H0: média_modernizados == média_ineficientes (não há diferença)
    # H1: média_modernizados != média_ineficientes (há diferença significativa)
    statistic, pvalue = stats.ttest_ind(amostra_mod, amostra_inef, alternative='two-sided');

    if verbose:
        print(f"Média Modernizados:  {amostra_mod.mean()}");
        print(f"Média Ineficientes:  {amostra_inef.mean()}");
        print(f"Estatística: {statistic}");
        print(f"p-value: {pvalue}");
        if pvalue < alpha:
            print("Rejeita-se H0: a ocupação média difere significativamente entre os dois grupos");
        else:
            print("Não se rejeita H0: não há evidência de diferença significativa entre os grupos");

    return pvalue >= alpha;

teste_ocupação_media_ef_vs_nef();

3- Considere três amostras aleatórias de 25 concelhos representativas de diferentes perfis de ocupação da rede: Norte/Centro Litoral (Porto, Braga, Coimbra), Lisboa e Litoral Sul (Lisboa, Setúbal, Aveiro) e Interior/Alentejo (Évora, Beja, Portalegre). Use estas amostras para testar a existência de diferenças significativas nos níveis médios de carga da rede (ANOVA). Caso necessário, efetue uma análise post-hoc adequada

In [ ]:
def teste_anova_regioes(s: int = 32, alpha: float = 0.05, verbose: bool = True) -> bool:
    """
    Testa se existem diferenças significativas no nível médio de ocupação da rede
    entre três regiões: Norte/Centro Litoral, Lisboa e Litoral Sul, e Interior/Alentejo.
    Usa amostras aleatórias de 25 concelhos por região.
    Retorna True se não se rejeita H0 (não há diferença), False caso contrário.
    Caso H0 seja rejeitado, aplica análise post-hoc de Tukey.
    """
    combined = resulting_data.join(ptd_stats['Util_Media']).reset_index();

    # Códigos de distrito por região
    distritos_norte_centro =    [6, 3, 13];   # Coimbra, Braga, Porto
    distritos_lisboa_litoral =  [11, 15, 1];  # Lisboa, Setúbal, Aveiro
    distritos_interior =        [7, 2, 12];   # Évora, Beja, Portalegre

    # Cada código de concelho começa com o código do distrito (primeiros 2 dígitos)
    norte_centro   = combined[(combined['CodDistritoConcelho'] // 100).isin(distritos_norte_centro)];
    lisboa_litoral = combined[(combined['CodDistritoConcelho'] // 100).isin(distritos_lisboa_litoral)];
    interior       = combined[(combined['CodDistritoConcelho'] // 100).isin(distritos_interior)];

    # Amostras de 25 concelhos por região
    amostra_norte =    norte_centro.sample(n=25, random_state=s)['Util_Media'];
    amostra_lisboa =   lisboa_litoral.sample(n=25, random_state=s)['Util_Media'];
    amostra_interior = interior.sample(n=25, random_state=s)['Util_Media'];

    # Verificar normalidade de cada grupo — ANOVA assume normalidade
    normal_norte    = test_normalidade(amostra_norte, nome="Norte/Centro Litoral", alpha=alpha, verbose=verbose);
    normal_lisboa   = test_normalidade(amostra_lisboa, nome="Lisboa e Litoral Sul", alpha=alpha, verbose=verbose);
    normal_interior = test_normalidade(amostra_interior,nome="Interior/Alentejo", alpha=alpha, verbose=verbose);

    if not normal_norte or not normal_lisboa or not normal_interior:
        raise ValueError("Os dados deixaram de ser normais!");

    # H0: média_norte == média_lisboa == média_interior (não há diferença entre regiões)
    # H1: pelo menos uma região difere significativamente das restantes
    result = stats.f_oneway(amostra_norte, amostra_lisboa, amostra_interior);

    if verbose:
        print(f"\nANOVA de um fator");
        print(f"Média Norte/Centro Litoral:  {amostra_norte.mean()}");
        print(f"Média Lisboa e Litoral Sul:  {amostra_lisboa.mean()}");
        print(f"Média Interior/Alentejo:     {amostra_interior.mean()}");
        print(f"Estatística F: {result.statistic}");
        print(f"p-value: {result.pvalue}");

        if result.pvalue < alpha:
            print("Rejeita-se H0: existem diferenças significativas entre as regiões");

            # Análise post-hoc de Tukey para identificar quais pares diferem
            valores  = np.concatenate([amostra_norte, amostra_lisboa, amostra_interior]);
            grupos   = (["Norte/Centro"] * len(amostra_norte) +
                        ["Lisboa/Litoral"] * len(amostra_lisboa) +
                        ["Interior"] * len(amostra_interior));

            tukey = pairwise_tukeyhsd(valores, grupos, alpha=alpha);
            print(f"\nAnálise Post-hoc de Tukey");
            print(tukey);
        else:
            print("Não se rejeita H0: não há evidência de diferenças significativas entre as regiões");

    return result.pvalue >= alpha;

teste_anova_regioes();

4- Teste a existência de uma relação linear estatisticamente significativa entre a capacidade total de transformação instalada e a carga de iluminação pública para a totalidade dos concelhos, interpretando o coeficiente de correlação de Pearson

In [ ]:
def teste_correlacao_pearson(alpha: float = 0.05, verbose: bool = True) -> bool:
    """
    Testa a existência de uma relação linear estatisticamente significativa
    entre a capacidade total de transformação instalada (Cap_PTD) e a carga
    de iluminação pública (P_IP_Total) para a totalidade dos concelhos.
    Retorna True se não se rejeita H0 (não há correlação), False caso contrário.
    """
    combined = result.join(ptd_stats['Cap_PTD']);

    cap_ptd   = combined['Cap_PTD'];
    p_ip_total = combined['P_IP_Total'];

    # H0: rho == 0 (não existe relação linear entre as variáveis)
    # H1: rho != 0 (existe relação linear estatisticamente significativa)
    statistic, pvalue = stats.pearsonr(cap_ptd, p_ip_total);

    if verbose:
        print(f"\nCorrelação de Pearson");
        print(f"Coeficiente r: {statistic}");
        print(f"p-value: {pvalue}");

        # Interpretar a força da correlação
        r = abs(statistic);
        if r < 0.2:
            forca = "muito fraca";
        elif r < 0.4:
            forca = "fraca";
        elif r < 0.6:
            forca = "moderada";
        elif r < 0.8:
            forca = "forte";
        else:
            forca = "muito forte";

        direcao = "positiva" if statistic > 0 else "negativa";
        print(f"Interpretação: correlação {forca} e {direcao}");

        if pvalue < alpha:
            print("Rejeita-se H0: existe relação linear significativa entre Cap_PTD e P_IP_Total");
        else:
            print("Não se rejeita H0: não há evidência de relação linear significativa");

    return pvalue >= alpha;

teste_correlacao_pearson();

#### Testar com várias seeds
A maneira como arquitecturamos estes testes (ver nota no topo do capítulo) vais ons permitir retestar tudo com várias seeds para ter uma melhor visão da variabilidade desta informação

In [ ]:
def testar_seeds(seeds: range = range(1, 101), alpha: float = 0.05) -> None:
    """
    Corre as três funções de teste para várias seeds e mostra um gráfico
    com os resultados (Rejeita/Não rejeita H0 / Dados não normais) para cada seed.
    """
    # None = dados não normais
    resultados = {
        "teste_ocupação_media_menor_que": [],
        "teste_ocupação_media_ef_vs_nef": [],
        "teste_anova_regioes": []
    };

    for s in seeds:
        for nome, func in [
            ("teste_ocupação_media_menor_que", lambda s: teste_ocupação_media_menor_que(s=s, alpha=alpha, verbose=False)),
            ("teste_ocupação_media_ef_vs_nef", lambda s: teste_ocupação_media_ef_vs_nef(s=s, alpha=alpha, verbose=False)),
            ("teste_anova_regioes",            lambda s: teste_anova_regioes(s=s, alpha=alpha, verbose=False)),
        ]:
            try:
                resultados[nome].append(func(s));
            except ValueError:
                resultados[nome].append(None); # dados não normais

    # Gráfico
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True);

    titulos = [
        "Ocupação média < 60%",
        "Modernizados vs Ineficientes",
        "ANOVA Regiões"
    ];

    def cor(v):
        if v is None: return "orange";
        return "green" if v else "red";

    for ax, (nome, valores), titulo in zip(axes, resultados.items(), titulos):
        cores = [cor(v) for v in valores];
        ax.bar(list(seeds), [1] * len(seeds), color=cores, width=0.8);
        ax.set_ylabel(titulo, fontsize=9);
        ax.set_yticks([]);
        ax.set_xlim(min(seeds) - 0.5, max(seeds) + 0.5);

    # Legenda
    from matplotlib.patches import Patch;
    legenda = [
        Patch(color="green",  label="Não rejeita H0"),
        Patch(color="red",    label="Rejeita H0"),
        Patch(color="orange", label="Dados não normais"),
    ];
    fig.legend(handles=legenda, loc="upper right");
    fig.suptitle("Resultados por Seed", fontsize=12);
    axes[-1].set_xlabel("Seed");
    plt.tight_layout();
    plt.show();

    # Resumo
    for nome, valores in resultados.items():
        total    = len(valores);
        rejeita  = sum(1 for v in valores if v == False and v is not None)
        nao_rej  = sum(1 for v in valores if v == True  and v is not None)
        nao_norm = sum(1 for v in valores if v is None)
        print(f"{nome}:");
        print(f"\tNão rejeita H0:   {nao_rej}/{total} ({nao_rej/total*100:.1f}%)");
        print(f"\tRejeita H0:       {rejeita}/{total} ({rejeita/total*100:.1f}%)");
        print(f"\tDados não normais:{nao_norm}/{total} ({nao_norm/total*100:.1f}%)\n");

testar_seeds();

### 4.5 - Correlação e Regressão - Modelo Preditivo

Considere os dados relativos aos níveis médios de carga dos PTDs nos distritos de Aveiro, Porto, Lisboa e Braga e construa uma tabela de correlação entre as principais métricas de infraestrutura (Capacidade PTD e Potência IP) para estes distritos. Selecione os dados relativos a Portugal Continental. Considere as seguintes variáveis explicativas:

- X1 – Potência Instalada Total de Iluminação Pública ***(P_IP_Total)***
- X2 – Capacidade Nominal de Transformação ***(Cap_PTD)***
- X3 – Ineficiência ***(Rate_Ineficencia)***

e a variável dependente:

- Y – Estado de Ocupação Médio da Rede ***(Util_Media)***.

1- Determine o modelo de regressão linear múltipla que explique a variação de Y em função de X1, X2 e X3.

In [ ]:
# 1 goes here

2- Verifique as condições sobre os resíduos (normalidade, independência e homocedasticidade).

In [ ]:
# 2 goes here

3- Verifique se existe multicolinearidade entre as variáveis recorrendo ao cálculo do VIF *(Variance Inflation Factor)*.

In [ ]:
# 3 goes here

4- Comente o modelo obtido tendo em conta o coeficiente de determinação ajustado e a significância estatística dos preditores.

In [ ]:
# 4 goes here

5- Estime o nível de ocupação esperado (Y) para os concelhos com os códigos: 101, 102, 103, 104, 105, 106, 107, 108 e 109 (Distrito de Aveiro) e compare as previsões com os valores reais presentes no dataset.

In [ ]:
# 5 goes here

6- Com base no coeficiente β3 obtido no modelo, quantifique a redução esperada no nível de ocupação da rede caso o rácio de ineficiência tecnológica seja reduzido em 20\% através da implementação de tecnologia LED.

In [ ]:
# 6 goes here

7- Utilize o modelo para identificar os concelhos onde a libertação de potência é estatisticamente mais viável para a instalação de carregadores de veículos elétricos de 22 kW, justificando com base nos intervalos de confiança das previsões.

In [ ]:
# 7 goes here

### 4.6 - Análise e Discussão de Resultados

Efetue uma síntese dos resultados e das conclusões, obtidos neste trabalho, que considera mais importantes, justificando sempre que necessário (conclusão). Use um nível de significância de 5% em todos os testes de hipótese que realizar.